# 01 - Exploratory Data Analysis & Canonical Normalisation

## Overview
This notebook loads raw football match event data from the `data/` directory (SPADL format), examines table schemas, distributions, and missing values, detects and fixes coordinate pitch mirroring (180° rotation), and derives fundamental match contexts (fixtures, goals, scorelines, own goals, player minutes, and positional centroids).

### Pipeline Steps:
1. **Load Raw Tables from `data/`**: `games.csv`, `teams.csv`, `players.csv`, `types.csv`, `results.csv`, `bodyparts.csv`, `actions.csv`.
2. **Inspect Coordinates & Invariants**: Validate spatial ranges ($x \in [0, 105], y \in [0, 68]$).
3. **Detect & Fix Pitch Inversion**: Invert attacking direction via 180° rotation ($x \to 105 - x, y \to 68 - y$).
4. **Reconstruct Match Context**: Match scorelines, goal difference per action, estimated minutes, and positional centroids.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = 'data'
PITCH_LENGTH = 105.0
PITCH_WIDTH = 68.0

print("Environment initialized.")


## 1. Relational Schema & Table Inspection in `data/`


In [ ]:
# Inspect available dimension tables in data/
for fname in ['games.csv', 'teams.csv', 'players.csv', 'types.csv', 'results.csv', 'bodyparts.csv']:
    fpath = os.path.join(DATA_DIR, fname)
    if os.path.exists(fpath):
        df_tmp = pd.read_csv(fpath)
        print(f"File: {fname:<15} | Shape: {df_tmp.shape} | Columns: {list(df_tmp.columns)}")


## 2. Load Actions & Inspect Spatial Distributions


In [ ]:
actions_path = os.path.join(DATA_DIR, 'actions.csv')
if not os.path.exists(actions_path):
    actions_path = 'actions.csv'

print(f"Loading action stream from: {actions_path}")
df_actions = pd.read_csv(actions_path, nrows=100000)
print("Action stream sample loaded:", df_actions.shape)
df_actions.head(5)


## 3. Pitch Inversion Detection & 180° Canonicalisation
Teams in raw data attack towards $x=0$. We apply a 180° rotation ($x \to 105 - x$, $y \to 68 - y$) to establish standard attacking geometry ($x \to 105$).


In [ ]:
def canonicalise_coordinates(df):
    df = df.copy()
    goal_kicks = df[df['type_id'] == 13] if 'type_id' in df.columns else df
    mean_gk_x = goal_kicks['start_x'].mean() if len(goal_kicks) > 0 else 50.0
    
    if mean_gk_x > 52.5:
        print(f"Detected inverted orientation (Mean GK x = {mean_gk_x:.2f}m). Applying 180° rotation...")
        df['start_x'] = PITCH_LENGTH - df['start_x']
        df['start_y'] = PITCH_WIDTH - df['start_y']
        df['end_x'] = PITCH_LENGTH - df['end_x']
        df['end_y'] = PITCH_WIDTH - df['end_y']
    return df

df_actions_canonical = canonicalise_coordinates(df_actions)
df_actions_canonical[['start_x', 'start_y', 'end_x', 'end_y']].describe()


## 4. Derive Match Scores, Game States & Playing Windows


In [ ]:
def derive_match_context(df):
    df = df.sort_values(['game_id', 'period_id', 'time_seconds' if 'time_seconds' in df.columns else 'timeSeconds'])
    
    # Identify goals
    is_shot_goal = (df['type_id'] == 11) & (df['result_id'] == 1)
    is_own_goal = (df['type_id'] == 18) & (df['result_id'] == 3)
    df['is_goal'] = is_shot_goal | is_own_goal
    
    print(f"Goals identified: {df['is_goal'].sum()}")
    return df

df_clean = derive_match_context(df_actions_canonical)
print("EDA and Data Cleaning stage successfully verified.")
